# Notebook 2: Building and Applying the Codebook

## Purpose
This notebook turns the candidate themes from Notebook 1 into a formal codebook
with clear definitions and boundary rules, then applies it to all 204 facilities-
feedback responses.

## Two decisions made before coding starts

These came up while testing the candidate themes against a second sample, and
I'm deciding them now, before coding the full dataset, so the rule is applied
consistently rather than made up response by response.

**1. Parking is being split into three sub-codes, not one.**
Testing showed "parking" responses actually fall into three distinct issues
that would need different fixes: not enough spaces, cost of parking, and
safety or layout problems getting in and out. Collapsing these into one code
would hide that distinction. So Parking becomes three codes:
- Parking: Availability
- Parking: Cost
- Parking: Safety/Layout

**2. General Satisfaction requires a reason. Bare thanks or approval with no
detail is coded separately as Non-substantive, not as Satisfaction.**
Responses like "Thanks!" or "All is good" don't tell us anything about what's
working, so treating them the same as "I love the outside area, it's great for
events" would overstate how much evidence I actually have for what people value.
Rule:
- **General Satisfaction** = positive AND names a specific reason (a space, a
  feature, staff, an experience)
- **Non-substantive** = too short or too vague to code for content, whether
  positive, negative, or unclear (e.g. "Thanks!", "NO LONGER THEME CENTER,
  WHY?", or pure profanity with no other content)

This means Non-substantive is a genuinely mixed bucket sentiment-wise. I'm
accepting that trade-off because the alternative, guessing what a one-word
response "really meant," would be worse.

In [2]:
# 1. Load the facilities-feedback data again in this notebook
# Each notebook loads its own data rather than relying on variables from
# a previous notebook still being in memory

import pandas as pd

df = pd.read_csv("../data/raw/City_Cultural_Centers_Audit_Community_Survey_-_Open_Response_Data_20260912.csv")
facilities_df = df[df["Survey Item"].str.contains("facilities", case=False)].copy()
facilities_df = facilities_df.reset_index(drop=True)

print(f"Loaded {len(facilities_df)} facilities-feedback responses")
facilities_df.head()

Loaded 204 facilities-feedback responses


,Facility,Survey Item,Response,Auditor-assigned Category,Re-assigned response?,Translated?,Original Language
0,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,TJ Ownes should be compensated on the level of...,NaN,False,False,English
1,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,I would like to see more professional producti...,NaN,False,False,English
2,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,"The dance studio floor is consistently dirty, ...",Negative,False,False,English
3,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,This facility needs some TLC. It needs renovat...,Negative,False,False,English
4,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,The facility needs to be updated and a deep cl...,Negative,False,False,English


## The Codebook

Ten codes. A response can receive multiple codes. Each code has a
one-line definition and a short rule for the boundary case that came up
most often while testing.

| Code | Definition | Boundary rule |
|---|---|---|
| **Space: Size/Capacity** | Complaint or comment about the physical space being too small, crowded, or unable to hold the intended activity or audience | Distinct from Maintenance: this is about the space being the wrong *size*, not in poor *condition* |
| **Maintenance/Condition** | The physical condition or provisioning of the space, including cleanliness, disrepair, and inadequate equipment or amenities (e.g., kitchen facilities, climate control). | If a response says something needs replacing or is broken, code here even if it also mentions size |
| **Parking: Availability** | Not enough parking spaces, or demand outstripping supply | |
| **Parking: Cost** | Parking is too expensive, or paid parking is a problem | |
| **Parking: Safety/Layout** | Getting in or out of parking is dangerous, confusing, or poorly designed (traffic risk, confusing exits, hazards) | |
| **Parking: External Pressure** | Parking is being used or affected by people attending nearby, unrelated venues rather than the cultural centre itself (e.g. bar patrons from Rainey Street occupying MACC spaces). Distinct from Parking: Availability, which is about there simply not being enough spaces or the centre's own users, regardless of cause.
| **Hours of Operation** | The facility's opening hours, days open, or time restrictions are a problem | Distinct from Accessibility: this is about *when* it's open, not *whether* someone can physically get there or use it |
| **Accessibility/Location** | Physical accessibility (wheelchair access, disability access) or geographic accessibility (too far, badly located relative to the community it serves) | |
| **Equity/Comparison** | Explicitly compares this facility's size, funding, or treatment to another facility or community, usually framed as unequal treatment | Code here even if the response also mentions size or condition, since the comparative claim is the important part |
| **Booking Policy/Eligibility** | Comments about who is allowed to book or use the space, and under what conditions (e.g. eligibility rules tied to a specific cultural affiliation, restrictions on what an event can be for). | Distinct from Space: Size/Capacity and Programming/Content, since this is about access rules and permissions, not the physical space or what's exhibited inside it. |
| **Programming/Content** | Requests or comments about what the facility hosts, exhibits, or represents (specific groups, exhibits, cultural content), as opposed to the physical space itself. Distinct from Space: Size/Capacity, which is about whether the space is big enough, not what it's used for |
| **General Satisfaction** | Positive sentiment naming the facility directly counts, even without one specific concrete feature (e.g. "I love this place," "Great facilities, very welcoming"). Originally, this code required a named reason; loosened after seeing how common simple, clear positive statements were, and that treating them as Non-substantive understated genuine satisfaction. Non-substantive is now reserved for responses too vague, factual, or off-topic to carry any evaluative content at all (positive, negative, or neutral). |
| **Non-substantive** | Too short, unclear, or off-topic to code for content: bare thanks, confusion, profanity with no other content, or comments not actually about facilities | Can be positive, negative, or neutral in tone; the point is there's nothing else to extract |


**Note on multiple codes:** most short responses will get one code. Longer
responses, like the AACHF dance studio complaint from Notebook 1, may get
four or five. This is expected and important; forcing one code per response
would lose real information.

In [4]:
# 2. Set up columns for each code
# Using one boolean column per code makes it easy to allow multiple codes
# per response, and easy to check counts later

code_columns = [
    "space_size",
    "maintenance",
    "parking_availability",
    "parking_cost",
    "parking_safety",
    "parking_external_pressure",  # added: parking pressure from nearby unrelated venues
    "hours",
    "accessibility_location",
    "equity_comparison",
    "general_satisfaction",
    "non_substantive",
    "programming_content",
    "booking_policy",
]

for col in code_columns:
    facilities_df[col] = False

In [5]:
# Add the new column without resetting existing coding
facilities_df["parking_external_pressure"] = False

# Also update code_columns list for later use in cell 6/analysis, without
# re-running the reset loop above
code_columns.append("parking_external_pressure")

## Trial run: coding a small subset first

Before coding all 204 responses, I'm testing the codebook against 20 responses
I haven't looked at yet. This checks two things: whether the ten codes are
enough to cover what's actually there, and whether I can apply them
consistently without needing to stop and rewrite definitions halfway through.

In [7]:
# 3. Pull a trial sample, excluding anything already read in NB01
# NB01's samples aren't in this notebook's memory, so I'm approximating by
# taking a fresh random sample; some overlap with NB01 is fine for a trial run

trial_sample = facilities_df.sample(20, random_state=99)
trial_sample[["Facility", "Response"]]

,Facility,Response
71,Emma S. Barrientos Mexican American Cultural C...,"as another City employee, I have had an awful ..."
24,Asian American Resource Center,Wonderful to have the AARC!
51,Asian American Resource Center,I am waiting for more mixed use space like stu...
162,George Washington Carver Museum,Sometimes parking is issue because of many att...
97,Emma S. Barrientos Mexican American Cultural C...,I can definitely take my car to the facility a...
40,Asian American Resource Center,The lunch room needs expansion.
124,Emma S. Barrientos Mexican American Cultural C...,"1) Rust, dirty exterior walls, peeling paint.\..."
84,Emma S. Barrientos Mexican American Cultural C...,The parking is horrible and there should be an...
142,Emma S. Barrientos Mexican American Cultural C...,Facility is small and a bit hard to access. Mu...
131,Emma S. Barrientos Mexican American Cultural C...,The space is so beautiful and it reminds the M...


In [8]:
# 4. Print the trial sample in full so nothing gets truncated
for i, row in trial_sample.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[71] --- Emma S. Barrientos Mexican American Cultural Center ---
as another City employee, I have had an awful time trying to book rooms for nature-based educational events for the community - was told it had to have a nexus with Mexican-American/Latino culture even though we were trying to engage more nearby neighbors in nature activities

[24] --- Asian American Resource Center ---
Wonderful to have the AARC!

[51] --- Asian American Resource Center ---
I am waiting for more mixed use space like studios and theater space.

[162] --- George Washington Carver Museum ---
Sometimes parking is issue because of many attendees.

[97] --- Emma S. Barrientos Mexican American Cultural Center ---
I can definitely take my car to the facility and I know some can get there by bike but the bus option is terrible. I really wish CapMetro would have better stop options for this site and Fiesta Gardens. It baffles me why I have to walk a mile with my 3 year old to get to a city cultural center.

[40] -

In [9]:
# 5. Manually code the trial sample
# Decisions applied per the boundary rules worked out during this trial:
# - Accessibility/Location broadened to include public transit access
# - Equity/Comparison broadened to include disrespect/unequal treatment claims,
#   not just facility-to-facility comparisons
# - General Satisfaction counts personal/cultural meaning as a valid "reason"
# - New code added: booking_policy, for eligibility/booking-rule complaints

facilities_df.loc[71, "booking_policy"] = True

facilities_df.loc[24, "non_substantive"] = True

facilities_df.loc[51, "space_size"] = True

facilities_df.loc[162, "parking_availability"] = True

facilities_df.loc[97, "accessibility_location"] = True

facilities_df.loc[40, "space_size"] = True

facilities_df.loc[124, "maintenance"] = True
facilities_df.loc[124, "equity_comparison"] = True

facilities_df.loc[84, "parking_availability"] = True

facilities_df.loc[142, "space_size"] = True
facilities_df.loc[142, "accessibility_location"] = True
facilities_df.loc[142, "equity_comparison"] = True

facilities_df.loc[131, "general_satisfaction"] = True

facilities_df.loc[133, "space_size"] = True

facilities_df.loc[167, "general_satisfaction"] = True

facilities_df.loc[150, "general_satisfaction"] = True
facilities_df.loc[150, "maintenance"] = True

facilities_df.loc[158, "general_satisfaction"] = True
facilities_df.loc[158, "space_size"] = True

facilities_df.loc[116, "parking_availability"] = True
facilities_df.loc[116, "general_satisfaction"] = True

facilities_df.loc[156, "space_size"] = True
facilities_df.loc[156, "parking_availability"] = True

facilities_df.loc[199, "non_substantive"] = True

facilities_df.loc[98, "non_substantive"] = True

facilities_df.loc[53, "hours"] = True

facilities_df.loc[194, "maintenance"] = True

In [10]:
# 6. Check trial coding results
trial_sample_coded = facilities_df.loc[trial_sample.index, ["Facility", "Response"] + code_columns]
trial_sample_coded

,Facility,Response,space_size,maintenance,parking_availability,parking_cost,parking_safety,parking_external_pressure,hours,accessibility_location,equity_comparison,general_satisfaction,non_substantive,programming_content,booking_policy,parking_external_pressure
71,Emma S. Barrientos Mexican American Cultural C...,"as another City employee, I have had an awful ...",False,False,False,False,False,False,False,False,False,False,False,False,True,False
24,Asian American Resource Center,Wonderful to have the AARC!,False,False,False,False,False,False,False,False,False,False,True,False,False,False
51,Asian American Resource Center,I am waiting for more mixed use space like stu...,True,False,False,False,False,False,False,False,False,False,False,False,False,False
162,George Washington Carver Museum,Sometimes parking is issue because of many att...,False,False,True,False,False,False,False,False,False,False,False,False,False,False
97,Emma S. Barrientos Mexican American Cultural C...,I can definitely take my car to the facility a...,False,False,False,False,False,False,False,True,False,False,False,False,False,False
40,Asian American Resource Center,The lunch room needs expansion.,True,False,False,False,False,False,False,False,False,False,False,False,False,False
124,Emma S. Barrientos Mexican American Cultural C...,"1) Rust, dirty exterior walls, peeling paint.\...",False,True,False,False,False,False,False,False,True,False,False,False,False,False
84,Emma S. Barrientos Mexican American Cultural C...,The parking is horrible and there should be an...,False,False,True,False,False,False,False,False,False,False,False,False,False,False
142,Emma S. Barrientos Mexican American Cultural C...,Facility is small and a bit hard to access. Mu...,True,False,False,False,False,False,False,True,True,False,False,False,False,False
131,Emma S. Barrientos Mexican American Cultural C...,The space is so beautiful and it reminds the M...,False,False,False,False,False,False,False,False,False,True,False,False,False,False


## Coding the full dataset

The codebook is now stable at 11 codes after one trial run. I'm coding all
204 facilities-feedback responses against it.

This is a manual process: reading each response and deciding which codes
apply, same as the trial. 

To make 204 responses manageable, I'll work through them in batches (of
around 25-30), printing a batch, coding it by hand, then moving to the next.
This also means if I need to stop and come back, I'm not starting from a
mid-batch position I can't reconstruct.

In [12]:
# 7. Set up batching so I can work through 204 responses in manageable chunks
# rather than trying to hold all of them in mind at once

batch_size = 25
all_indices = facilities_df.index.tolist()
n_batches = (len(all_indices) // batch_size) + 1

print(f"{len(all_indices)} responses, {n_batches} batches of up to {batch_size}")

204 responses, 9 batches of up to 25


In [13]:
# 8. Print a batch to read and code
# Change batch_number each time you move to the next batch: 0, 1, 2, ...

batch_number = 0  # change this each time

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[0] --- African American Cultural and Heritage Facility ---
TJ Ownes should be compensated on the level of all other staff that heads and manage other Cultural facilities on the behalf of the city of Austin.

[1] --- African American Cultural and Heritage Facility ---
I would like to see more professional productions in the theater. I would like Spectrum Theater Group to have the space as their home space.

[2] --- African American Cultural and Heritage Facility ---
The dance studio floor is consistently dirty, often with debris on the floor which can be hazardous to dancers' feet (small pebbles; I've even found curtain rod hooks and metal wires ON THE FLOOR). The carpet runners provided from the time the facility opened appear to have not been cleaned since they were installed; I have to sweep the floor every time I use the facility. The bathroom is consistently out of soap; the toilet has a nasty "ring" in the bowl, as if it hasn't been scrubbed in years, and the floor does not appea

In [14]:
# 9. Code batch 0

facilities_df.loc[0, "non_substantive"] = True  # staff compensation, not facility

facilities_df.loc[1, "programming_content"] = True  # NEW CODE: wants specific programming/theatre group hosted

facilities_df.loc[2, "space_size"] = True
facilities_df.loc[2, "maintenance"] = True
facilities_df.loc[2, "equity_comparison"] = True

facilities_df.loc[3, "maintenance"] = True
facilities_df.loc[3, "space_size"] = True

facilities_df.loc[4, "maintenance"] = True
facilities_df.loc[4, "general_satisfaction"] = True  # names location as a reason

facilities_df.loc[5, "programming_content"] = True  # NEW CODE: wants representational content/exhibits

facilities_df.loc[6, "maintenance"] = True

facilities_df.loc[7, "non_substantive"] = True

facilities_df.loc[8, "accessibility_location"] = True

facilities_df.loc[9, "space_size"] = True
facilities_df.loc[9, "general_satisfaction"] = True  # names location as a reason

facilities_df.loc[10, "hours"] = True

facilities_df.loc[11, "space_size"] = True

facilities_df.loc[12, "equity_comparison"] = True

facilities_df.loc[13, "non_substantive"] = True  # political statement, not a facility complaint

facilities_df.loc[14, "parking_availability"] = True

facilities_df.loc[15, "space_size"] = True

facilities_df.loc[16, "parking_availability"] = True

facilities_df.loc[17, "general_satisfaction"] = True  # specific: mother's room accommodation

facilities_df.loc[18, "parking_safety"] = True  # confusing/hazardous route from parking to building
facilities_df.loc[18, "general_satisfaction"] = True  # enjoys the events
facilities_df.loc[18, "space_size"] = True  # space a little small

facilities_df.loc[19, "general_satisfaction"] = True

facilities_df.loc[20, "space_size"] = True

facilities_df.loc[21, "parking_availability"] = True
facilities_df.loc[21, "accessibility_location"] = True  # transit/bus routes
facilities_df.loc[21, "general_satisfaction"] = True  # well-maintained
facilities_df.loc[21, "maintenance"] = True  # wants money for upgrades/restoration
facilities_df.loc[21, "space_size"] = True  # excited for expansion, real theatre

facilities_df.loc[22, "general_satisfaction"] = True

facilities_df.loc[23, "space_size"] = True

facilities_df.loc[24, "non_substantive"] = True

In [15]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 1  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[25] --- Asian American Resource Center ---
Austin has extremely very poor transportation for a city of its size

[26] --- Asian American Resource Center ---
(1) The parking can be tough especially the walk from overflow parking.

(2) I love the center.  I just wish it weren’t so far away.

[27] --- Asian American Resource Center ---
Regarding parking, I was attending an event with family and friends. Because the event was so popular, we had to park in the overflow parking area which was fine, but this was an example of how much more space the Center would need in the future. There is certainly room for growth and expansion.

[28] --- Asian American Resource Center ---
We have leased rooms a few times with no problems, had kind staff, and very helpful.

[29] --- Asian American Resource Center ---
The non-driving options to get to the facility aren't great. It's practically across the street from the Rundberg high-frequency transit corridor, but getting that last little bit is hard and 

In [16]:
# 9. Code batch 1

facilities_df.loc[25, "accessibility_location"] = True  # transit/transportation

facilities_df.loc[26, "parking_availability"] = True  # walk from overflow parking
facilities_df.loc[26, "general_satisfaction"] = True  # loves the center
facilities_df.loc[26, "accessibility_location"] = True  # wishes it weren't so far

facilities_df.loc[27, "parking_availability"] = True
facilities_df.loc[27, "space_size"] = True  # room for growth and expansion

facilities_df.loc[28, "general_satisfaction"] = True

facilities_df.loc[29, "accessibility_location"] = True  # transit access

facilities_df.loc[30, "non_substantive"] = True  # positive, no specific reason

facilities_df.loc[31, "parking_safety"] = True

facilities_df.loc[32, "programming_content"] = True  # requests new center location, fee comparison
facilities_df.loc[32, "accessibility_location"] = True  # requests NW Austin location
facilities_df.loc[32, "maintenance"] = True  # booking cleanup burden on renters

facilities_df.loc[33, "accessibility_location"] = True  # wants more central location
facilities_df.loc[33, "space_size"] = True  # wants a proper performance space

facilities_df.loc[34, "parking_availability"] = True

facilities_df.loc[35, "general_satisfaction"] = True  # parking well sign-posted

facilities_df.loc[36, "space_size"] = True

facilities_df.loc[37, "space_size"] = True  # demand outstrips supply of bookable space

facilities_df.loc[38, "space_size"] = True  # classrooms, ballroom, meeting space too limited
facilities_df.loc[38, "maintenance"] = True  # ants, weeds, signage, atmosphere
facilities_df.loc[38, "general_satisfaction"] = True  # garden, exercise equipment, educational programs
facilities_df.loc[38, "programming_content"] = True  # more diverse cultures promoted
facilities_df.loc[38, "parking_availability"] = True
facilities_df.loc[38, "hours"] = True  # evening and weekend hours

facilities_df.loc[39, "space_size"] = True

facilities_df.loc[40, "space_size"] = True

facilities_df.loc[41, "programming_content"] = True  # requests specific room/activity type

facilities_df.loc[42, "parking_availability"] = True

facilities_df.loc[43, "hours"] = True

facilities_df.loc[44, "parking_safety"] = True

facilities_df.loc[45, "space_size"] = True

facilities_df.loc[46, "space_size"] = True  # bigger rooms needed
                                              # NOTE: food/catering complaint left uncoded,
                                              # doesn't fit any of 13 codes, flagged as outlier

facilities_df.loc[47, "non_substantive"] = True

facilities_df.loc[48, "programming_content"] = True  # landscaping/garden features request

facilities_df.loc[49, "parking_safety"] = True

In [17]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 2  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[50] --- Asian American Resource Center ---
Please ask Capital Metro to move their bus stop closer to the AARC. It is a very long walk from both directions.

[51] --- Asian American Resource Center ---
I am waiting for more mixed use space like studios and theater space.

[52] --- Asian American Resource Center ---
(1) I am so very glad to have that resource in the   District. Visit Austin and the Center need to insure that visitors who might not venture outside of downtown, know that the center exists and the transit routes that can get them from their hotel to the center.

(2) Parking is inadequate. The need to construct a bridge across the gully from the city's parking lot to the grounds of the Center has been in the works for years.

[53] --- Asian American Resource Center ---
It would be nice for the AARC's later evening hours to be restored on Mondays and Tuesdays, so the facility can be open to the public four nights a week and so that groups such as ACC can offer more ESL class

In [18]:
# 9. Code batch 2
# Maintenance/Condition definition widened this batch to cover kitchen
# adequacy and climate control, not just cleanliness/disrepair

facilities_df.loc[50, "accessibility_location"] = True  # transit stop distance

facilities_df.loc[51, "space_size"] = True

facilities_df.loc[52, "programming_content"] = True  # visibility/awareness of the center
facilities_df.loc[52, "accessibility_location"] = True  # transit routes from downtown
facilities_df.loc[52, "parking_availability"] = True
facilities_df.loc[52, "parking_safety"] = True  # bridge across the gully

facilities_df.loc[53, "hours"] = True

facilities_df.loc[54, "parking_availability"] = True
facilities_df.loc[54, "maintenance"] = True  # rough grounds for tai chi

facilities_df.loc[55, "maintenance"] = True  # kitchen inadequate

facilities_df.loc[56, "space_size"] = True
facilities_df.loc[56, "programming_content"] = True  # more cultural programs

facilities_df.loc[57, "space_size"] = True

facilities_df.loc[58, "space_size"] = True

facilities_df.loc[59, "non_substantive"] = True  # "see above" carries nothing alone
facilities_df.loc[59, "space_size"] = True
facilities_df.loc[59, "maintenance"] = True  # no professional kitchen

facilities_df.loc[60, "space_size"] = True

facilities_df.loc[61, "space_size"] = True
facilities_df.loc[61, "general_satisfaction"] = True

facilities_df.loc[62, "space_size"] = True

facilities_df.loc[63, "programming_content"] = True  # more events
facilities_df.loc[63, "parking_availability"] = True

facilities_df.loc[64, "parking_availability"] = True

facilities_df.loc[65, "space_size"] = True
facilities_df.loc[65, "parking_availability"] = True

facilities_df.loc[66, "general_satisfaction"] = True  # clean, easy to work with

facilities_df.loc[67, "general_satisfaction"] = True

facilities_df.loc[68, "general_satisfaction"] = True

facilities_df.loc[69, "parking_availability"] = True

facilities_df.loc[70, "maintenance"] = True  # climate control (heat) issue

facilities_df.loc[71, "booking_policy"] = True

facilities_df.loc[72, "non_substantive"] = True  # comment on survey question design, not the facility

facilities_df.loc[73, "parking_availability"] = True
facilities_df.loc[73, "hours"] = True
facilities_df.loc[73, "maintenance"] = True  # office mess, bathroom issue, landscaping

facilities_df.loc[74, "maintenance"] = True

In [19]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 3  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[75] --- Emma S. Barrientos Mexican American Cultural Center ---
Parking and traffic in that area, especially on weekends with all the activity at the clubs and restaurants on Rainey street can be an issue.

[76] --- Emma S. Barrientos Mexican American Cultural Center ---
Replace elevators with newer safe elevators.

[77] --- Emma S. Barrientos Mexican American Cultural Center ---
Always clean and welcoming

[78] --- Emma S. Barrientos Mexican American Cultural Center ---
Sharing parking with Raini st. is not a good combination, during events, patrons and artists are getting parking tickets because the MACC does not provide parking attendants to some groups, so patrons have to pick up their parking slip at the office, risking getting a ticket while they are away from the car. There should be a better way and coordination with parking administration to consider hard working artists getting penalized .  We also need a bigger performing arts theater, the auditorium is too small for some e

In [20]:
# 9. Code batch 3

facilities_df.loc[75, "parking_safety"] = True  # traffic/congestion issue, not pure availability

facilities_df.loc[76, "maintenance"] = True

facilities_df.loc[77, "general_satisfaction"] = True

facilities_df.loc[78, "parking_safety"] = True  # ticketing risk, coordination failure
facilities_df.loc[78, "space_size"] = True  # auditorium too small

facilities_df.loc[79, "accessibility_location"] = True  # commute harder due to redevelopment
facilities_df.loc[79, "parking_safety"] = True  # construction debris, flat tire

facilities_df.loc[80, "non_substantive"] = True

facilities_df.loc[81, "general_satisfaction"] = True  # no specific reason beyond "nice"; borderline, see note below

facilities_df.loc[82, "equity_comparison"] = True  # who the facility is really for vs who's nearby
facilities_df.loc[82, "general_satisfaction"] = True  # loves it, supports it with tax dollars

facilities_df.loc[83, "hours"] = True

facilities_df.loc[84, "parking_availability"] = True

facilities_df.loc[85, "non_substantive"] = True  # explicit non-response, hasn't used the space

facilities_df.loc[86, "programming_content"] = True  # coffee shop / seating amenity request

facilities_df.loc[87, "parking_safety"] = True  # confusing, not unavailable

facilities_df.loc[88, "parking_availability"] = True
facilities_df.loc[88, "accessibility_location"] = True  # bus stop distance, mobility issues

facilities_df.loc[89, "parking_availability"] = True
facilities_df.loc[89, "space_size"] = True  # stages limited

facilities_df.loc[90, "maintenance"] = True  # outdated equipment

facilities_df.loc[91, "space_size"] = True  # not enough sitting/meeting area
facilities_df.loc[91, "maintenance"] = True  # "unorganized" reads as upkeep/management issue

facilities_df.loc[92, "space_size"] = True
facilities_df.loc[92, "parking_availability"] = True

facilities_df.loc[93, "general_satisfaction"] = True

facilities_df.loc[94, "maintenance"] = True

facilities_df.loc[95, "non_substantive"] = True  # positive, no specific reason

facilities_df.loc[96, "accessibility_location"] = True

facilities_df.loc[97, "accessibility_location"] = True

facilities_df.loc[98, "non_substantive"] = True  # hedges, doesn't commit to a position

facilities_df.loc[99, "general_satisfaction"] = True  # loves the place, names it directly

In [21]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 4  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[100] --- Emma S. Barrientos Mexican American Cultural Center ---
There is a great need to continue the expansion of the MACC in terms of studio, exhibition and performance space.  The city should proceed with the newly revised masterplan immediately.

[101] --- Emma S. Barrientos Mexican American Cultural Center ---
Public parking fees are to high and need to are more parking spots.

[102] --- Emma S. Barrientos Mexican American Cultural Center ---
The lot in front of the MACC which I believe they own, is full of weeds and dirt. Thats the intro to the MACC. And the trail behind it now has tents and garbage with the homeless. MACC leadership should take action. Its an embarrassment to the MA community. Again, great building and location but currently wasted potential.

[103] --- Emma S. Barrientos Mexican American Cultural Center ---
Additional parking is needed at ESB-MACC on scheduled events. Suggest working with Capitol Metro to provide bus service from a designated parking lot to/f

In [22]:
# 9. Code batch 4
# Assuming General Satisfaction loosened to accept clear positive sentiment
# naming the facility, even without one specific concrete feature
# parking_external_pressure used for rows where the pressure comes from
# nearby unrelated venues (Rainey Street), not just general shortage

facilities_df.loc[100, "space_size"] = True

facilities_df.loc[101, "parking_cost"] = True
facilities_df.loc[101, "parking_availability"] = True

facilities_df.loc[102, "maintenance"] = True

facilities_df.loc[103, "parking_availability"] = True

facilities_df.loc[104, "general_satisfaction"] = True

facilities_df.loc[105, "parking_external_pressure"] = True

facilities_df.loc[106, "accessibility_location"] = True

facilities_df.loc[107, "space_size"] = True
facilities_df.loc[107, "accessibility_location"] = True

facilities_df.loc[108, "maintenance"] = True
facilities_df.loc[108, "general_satisfaction"] = True

facilities_df.loc[109, "accessibility_location"] = True

facilities_df.loc[110, "non_substantive"] = True

facilities_df.loc[111, "general_satisfaction"] = True

facilities_df.loc[112, "general_satisfaction"] = True

facilities_df.loc[113, "general_satisfaction"] = True

facilities_df.loc[114, "general_satisfaction"] = True
facilities_df.loc[114, "parking_availability"] = True

facilities_df.loc[115, "non_substantive"] = True

facilities_df.loc[116, "parking_availability"] = True
facilities_df.loc[116, "general_satisfaction"] = True

facilities_df.loc[117, "space_size"] = True

facilities_df.loc[118, "parking_external_pressure"] = True
facilities_df.loc[118, "equity_comparison"] = True

facilities_df.loc[119, "maintenance"] = True
facilities_df.loc[119, "parking_external_pressure"] = True

facilities_df.loc[120, "parking_external_pressure"] = True
facilities_df.loc[120, "programming_content"] = True

facilities_df.loc[121, "maintenance"] = True

facilities_df.loc[122, "general_satisfaction"] = True

facilities_df.loc[123, "equity_comparison"] = True
facilities_df.loc[123, "hours"] = True
facilities_df.loc[123, "space_size"] = True
facilities_df.loc[123, "programming_content"] = True

facilities_df.loc[124, "maintenance"] = True
facilities_df.loc[124, "equity_comparison"] = True

In [41]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 5  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[125] --- Emma S. Barrientos Mexican American Cultural Center ---
All is good.

[126] --- Emma S. Barrientos Mexican American Cultural Center ---
It would be nice to have some more kid-friendly pieces inside and outside for kids to climb on and engage with.  It's a little sterile.

[127] --- Emma S. Barrientos Mexican American Cultural Center ---
The MACC needs to be expanded- larger gallery, cafe and more classrooms and meeting rooms

[128] --- Emma S. Barrientos Mexican American Cultural Center ---
I know there is a push to enlarge these facilities, however I don't believe that this need is greater than some of the other social needs our City faces like homelessness or actually providing affordable housing to these undeserved demographic groups. Once you enlarge one facility all the others are going to want to be enlarged too. This takes up a finite amount of resources and maintains the status quo, which is unacceptable.

[129] --- Emma S. Barrientos Mexican American Cultural Center 

In [43]:
# 9. Code batch 5

facilities_df.loc[125, "non_substantive"] = True

facilities_df.loc[126, "programming_content"] = True  # kid-friendly features/exhibits request

facilities_df.loc[127, "space_size"] = True

facilities_df.loc[128, "non_substantive"] = True
# FLAGGED: doesn't really fit. This is a policy argument against facility
# expansion generally, prioritising other social needs (housing, homelessness).
# Operates at a different level than the coding frame. Consider noting as a
# named example in limitations rather than forcing a fit.

facilities_df.loc[129, "parking_cost"] = True

facilities_df.loc[130, "non_substantive"] = True
# FLAGGED: "never felt welcome" is substantive but doesn't fit any current
# code. Watch for recurrence; may need an Atmosphere/Inclusion code if this
# pattern repeats.

facilities_df.loc[131, "general_satisfaction"] = True

facilities_df.loc[132, "general_satisfaction"] = True

facilities_df.loc[133, "space_size"] = True

facilities_df.loc[134, "programming_content"] = True  # studio access request

facilities_df.loc[135, "parking_external_pressure"] = True  # bar crowd, guards blocking patrons
facilities_df.loc[135, "accessibility_location"] = True  # handicap tag denied entry

facilities_df.loc[136, "general_satisfaction"] = True
facilities_df.loc[136, "accessibility_location"] = True  # neighbourhood safety concern

facilities_df.loc[137, "general_satisfaction"] = True
facilities_df.loc[137, "space_size"] = True  # at full capacity

facilities_df.loc[138, "parking_availability"] = True

facilities_df.loc[139, "maintenance"] = True  # layout/design complaint about bathroom placement

facilities_df.loc[140, "space_size"] = True

facilities_df.loc[141, "parking_availability"] = True

facilities_df.loc[142, "space_size"] = True
facilities_df.loc[142, "accessibility_location"] = True
facilities_df.loc[142, "equity_comparison"] = True

facilities_df.loc[143, "parking_cost"] = True
facilities_df.loc[143, "equity_comparison"] = True  # explicitly compares to other centers

facilities_df.loc[144, "non_substantive"] = True  # political statement, not a facility comment

facilities_df.loc[145, "general_satisfaction"] = True

facilities_df.loc[146, "parking_external_pressure"] = True  # Rainey residents using event parking
facilities_df.loc[146, "equity_comparison"] = True  # disrespect toward the community the center serves

facilities_df.loc[147, "maintenance"] = True  # elevator
facilities_df.loc[147, "programming_content"] = True  # marketing/awareness request

facilities_df.loc[148, "space_size"] = True
facilities_df.loc[148, "parking_cost"] = True
facilities_df.loc[148, "maintenance"] = True  # parking lot condition, gating

facilities_df.loc[149, "accessibility_location"] = True

In [45]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 6  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[150] --- Emma S. Barrientos Mexican American Cultural Center ---
Great building and location. They should move forward with their renovation. Could and should be a much better venue for showcasing MA culture.

[151] --- Emma S. Barrientos Mexican American Cultural Center ---
I have found it incredibly offensive to have seen dogs (off leash) and scooters go through the plaza at the MACC. I understand that there are trails and appartment complexes nearby, but these places should have fenced-in areas for dogs and scooters should be banned from roaming a cultural center.

[152] --- Emma S. Barrientos Mexican American Cultural Center ---
parking! wayfinding   (outside and inside) could not be more obtuse

[153] --- Emma S. Barrientos Mexican American Cultural Center ---
The ESB-MACC is at full capacity. The center needs to grow to keep up with the demand.

[154] --- Emma S. Barrientos Mexican American Cultural Center ---
More parking

[155] --- Emma S. Barrientos Mexican American Cultural 

In [47]:
# 9. Code batch 6

facilities_df.loc[150, "general_satisfaction"] = True
facilities_df.loc[150, "maintenance"] = True  # renovation needed
facilities_df.loc[150, "programming_content"] = True  # showcasing MA culture better

facilities_df.loc[151, "maintenance"] = True  # grounds/plaza condition and safety
facilities_df.loc[151, "equity_comparison"] = True  # disrespect of the cultural space, same theme as [146]

facilities_df.loc[152, "parking_availability"] = True
facilities_df.loc[152, "maintenance"] = True  # wayfinding/signage

facilities_df.loc[153, "space_size"] = True

facilities_df.loc[154, "parking_availability"] = True

facilities_df.loc[155, "parking_availability"] = True

facilities_df.loc[156, "space_size"] = True
facilities_df.loc[156, "parking_availability"] = True

facilities_df.loc[157, "parking_cost"] = True

facilities_df.loc[158, "general_satisfaction"] = True
facilities_df.loc[158, "space_size"] = True

facilities_df.loc[159, "parking_availability"] = True

facilities_df.loc[160, "general_satisfaction"] = True
facilities_df.loc[160, "accessibility_location"] = True  # central location valued

facilities_df.loc[161, "parking_availability"] = True

facilities_df.loc[162, "parking_availability"] = True

facilities_df.loc[163, "non_substantive"] = True  # unclear, no facility content extractable

facilities_df.loc[164, "maintenance"] = True
facilities_df.loc[164, "general_satisfaction"] = True

facilities_df.loc[165, "general_satisfaction"] = True
facilities_df.loc[165, "maintenance"] = True  # landscaping neglected

facilities_df.loc[166, "general_satisfaction"] = True

facilities_df.loc[167, "general_satisfaction"] = True  # personal/cultural meaning counts as a reason

facilities_df.loc[168, "maintenance"] = True
facilities_df.loc[168, "space_size"] = True
facilities_df.loc[168, "programming_content"] = True  # more options for classes/events

facilities_df.loc[169, "hours"] = True

facilities_df.loc[170, "general_satisfaction"] = True

facilities_df.loc[171, "parking_availability"] = True
facilities_df.loc[171, "maintenance"] = True  # signage for child safety
facilities_df.loc[171, "hours"] = True
facilities_df.loc[171, "programming_content"] = True  # evening event idea, referencing the Blanton

facilities_df.loc[172, "maintenance"] = True

facilities_df.loc[173, "general_satisfaction"] = True

facilities_df.loc[174, "hours"] = True
facilities_df.loc[174, "parking_cost"] = True  # "reasonable rates"

In [49]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 7  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[175] --- George Washington Carver Museum ---
The theater is need of a makeover, with redesign in mind. Seating chairs are torn, sound table in the wrong location and easier accessibility to lighting. A budget for maintenance would be helpful

[176] --- George Washington Carver Museum ---
Just dirty on stage and audience seats area.  not tuned piano provided (the pianist had to pay by her own account.)

[177] --- George Washington Carver Museum ---
Thanks!

[178] --- George Washington Carver Museum ---
This is an exceptional place. Thank You, the City of Austin.

[179] --- George Washington Carver Museum ---
museum, theater, bathrooms and kitchen are are dirty- Evidence of rats and cockroaches - They need professional cleaners!!!

[180] --- George Washington Carver Museum ---
The history of the Carver is very visible within the confines of the building.

[181] --- George Washington Carver Museum ---
1.) An expansion would be nice!
2.) The black community was short-changed when it came 

In [51]:
# 9. Code batch 7

facilities_df.loc[175, "maintenance"] = True

facilities_df.loc[176, "maintenance"] = True  # dirty, untuned piano as equipment issue

facilities_df.loc[177, "non_substantive"] = True  # bare thanks, no reason given

facilities_df.loc[178, "general_satisfaction"] = True  # names the place, clear sentiment, per revised rule

facilities_df.loc[179, "maintenance"] = True

facilities_df.loc[180, "general_satisfaction"] = True  # names a specific reason: visible history

facilities_df.loc[181, "space_size"] = True
facilities_df.loc[181, "equity_comparison"] = True

facilities_df.loc[182, "space_size"] = True  # classrooms/meeting rooms
facilities_df.loc[182, "accessibility_location"] = True  # transport for off-site programs
facilities_df.loc[182, "hours"] = True
facilities_df.loc[182, "maintenance"] = True  # understaffed cleaning, though praised

facilities_df.loc[183, "maintenance"] = True  # lighting near parking

facilities_df.loc[184, "space_size"] = True

facilities_df.loc[185, "accessibility_location"] = True  # wheelchair-accessible transport

facilities_df.loc[186, "hours"] = True

facilities_df.loc[187, "space_size"] = True
facilities_df.loc[187, "accessibility_location"] = True  # dangerous for elderly citizens

facilities_df.loc[188, "maintenance"] = True  # seats falling apart
facilities_df.loc[188, "programming_content"] = True  # underutilised for intended purpose

facilities_df.loc[189, "general_satisfaction"] = True  # accessibility praised
facilities_df.loc[189, "space_size"] = True  # wishes for expansion

facilities_df.loc[190, "space_size"] = True

facilities_df.loc[191, "space_size"] = True

facilities_df.loc[192, "maintenance"] = True

facilities_df.loc[193, "maintenance"] = True
# FLAGGED: "poorly run," no strategic plan, understaffed genealogy center.
# Governance/management critique, not purely physical. Watch if this recurs.

facilities_df.loc[194, "maintenance"] = True  # roof, storage
facilities_df.loc[194, "space_size"] = True  # phase II expansion question

facilities_df.loc[195, "equity_comparison"] = True

facilities_df.loc[196, "maintenance"] = True  # lighting

facilities_df.loc[197, "accessibility_location"] = True  # bus service praised

facilities_df.loc[198, "general_satisfaction"] = True
facilities_df.loc[198, "equity_comparison"] = True  # explicit comparison to other centers, positive direction

facilities_df.loc[199, "non_substantive"] = True

In [53]:
# 8. Print a batch to read and code
# Change batch_number each time I move to the next batch: 0, 1, 2, ...

batch_number = 8  # moved to next batch

start = batch_number * batch_size
end = start + batch_size
batch = facilities_df.iloc[start:end]

for i, row in batch.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[200] --- George Washington Carver Museum ---
The place is beautiful. Could use more plants in the court yard.

[201] --- George Washington Carver Museum ---
Well run with good facilities.

[202] --- George Washington Carver Museum ---
I believe the Museum is a great facility, but the childrens section is old and outdated.

[203] --- George Washington Carver Museum ---
nice meeting room for community agencies



In [55]:
# 9. Code batch 8 (final batch)

facilities_df.loc[200, "general_satisfaction"] = True
facilities_df.loc[200, "programming_content"] = True  # more plants/landscaping in courtyard

facilities_df.loc[201, "general_satisfaction"] = True

facilities_df.loc[202, "general_satisfaction"] = True  # great facility overall
facilities_df.loc[202, "maintenance"] = True  # children's section old/outdated

facilities_df.loc[203, "general_satisfaction"] = True

In [57]:
# 10. Sanity check: confirm every response has at least one code
# A response with all False across every column would mean something slipped
# through uncoded, worth catching now rather than during analysis

facilities_df["any_code"] = facilities_df[code_columns].any(axis=1)
uncoded = facilities_df[~facilities_df["any_code"]]

print(f"Responses with at least one code: {facilities_df['any_code'].sum()} / {len(facilities_df)}")
print(f"Uncoded responses: {len(uncoded)}")
uncoded[["Facility", "Response"]]

Responses with at least one code: 204 / 204
Uncoded responses: 0


,Facility,Response
